In [2]:
# ============================================================
# CODEBERT + PRIMEVUL : ONE-CELL FINAL WORKING CODE (KAGGLE)
# ============================================================

# --------------------
# Imports
# --------------------
import os
import json
import torch
import numpy as np
import pandas as pd
from datetime import datetime

from datasets import Dataset
from transformers import (
    RobertaTokenizer,
    RobertaForSequenceClassification,
    TrainingArguments,
    Trainer
)

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix
)

# --------------------
# DEVICE
# --------------------
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

# ============================================================
# DATASET PATH (✅ FIXED)
# ============================================================

DATASET_PATH = "/kaggle/input/primevul-dataset"

print("\nFiles in dataset:")
for f in os.listdir(DATASET_PATH):
    print(" -", f)

# ============================================================
# LOAD PRIMEVUL JSONL
# ============================================================

def load_primevul_jsonl(path):
    data = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            try:
                obj = json.loads(line.strip())
                data.append({
                    "code": obj["func"],
                    "label": int(obj["target"])
                })
            except Exception:
                continue
    return pd.DataFrame(data)

print("\nLoading PrimeVul dataset...")

train_df = load_primevul_jsonl(f"{DATASET_PATH}/primevul_train_paired.jsonl")
val_df   = load_primevul_jsonl(f"{DATASET_PATH}/primevul_valid_paired.jsonl")
test_df  = load_primevul_jsonl(f"{DATASET_PATH}/primevul_test_paired.jsonl")

print("Train:", train_df.shape)
print("Val  :", val_df.shape)
print("Test :", test_df.shape)
print("\nTrain label distribution:\n", train_df["label"].value_counts())

# ============================================================
# HF DATASETS
# ============================================================

train_ds = Dataset.from_pandas(train_df, preserve_index=False)
val_ds   = Dataset.from_pandas(val_df, preserve_index=False)
test_ds  = Dataset.from_pandas(test_df, preserve_index=False)

# ============================================================
# TOKENIZER
# ============================================================

tokenizer = RobertaTokenizer.from_pretrained("microsoft/codebert-base")

def tokenize_fn(batch):
    return tokenizer(
        batch["code"],
        truncation=True,
        padding="max_length",
        max_length=256
    )

train_ds = train_ds.map(tokenize_fn, batched=True)
val_ds   = val_ds.map(tokenize_fn, batched=True)
test_ds  = test_ds.map(tokenize_fn, batched=True)

cols = ["input_ids", "attention_mask", "label"]
train_ds.set_format("torch", columns=cols)
val_ds.set_format("torch", columns=cols)
test_ds.set_format("torch", columns=cols)

# ============================================================
# MODEL
# ============================================================

model = RobertaForSequenceClassification.from_pretrained(
    "microsoft/codebert-base",
    num_labels=2
).to(device)

# ============================================================
# TRAINING ARGUMENTS (NO CHECKPOINT SAVING)
# ============================================================

training_args = TrainingArguments(
    output_dir="./codebert_primevul",
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=6,
    learning_rate=2e-5,
    weight_decay=0.01,
    fp16=True,
    save_strategy="no",        # 🔥 prevents disk crash
    logging_steps=100,
    report_to="none"
)

# ============================================================
# TRAINER
# ============================================================

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds
)

# ============================================================
# TRAIN
# ============================================================

print("\nStarting CodeBERT training on PrimeVul...")
trainer.train()

# ============================================================
# FINAL EVALUATION
# ============================================================

print("\nEvaluating on test set...")

preds = trainer.predict(test_ds)

logits = preds.predictions
if isinstance(logits, tuple):   # safety
    logits = logits[0]

y_true = preds.label_ids
y_pred = np.argmax(logits, axis=1)

tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
fpr = fp / (fp + tn) if (fp + tn) > 0 else 0.0

accuracy  = accuracy_score(y_true, y_pred)
precision = precision_score(y_true, y_pred, zero_division=0)
recall    = recall_score(y_true, y_pred, zero_division=0)
f1        = f1_score(y_true, y_pred, zero_division=0)

print("\n===== FINAL CODEBERT PRIMEVUL RESULTS =====")
print("Accuracy :", accuracy)
print("Precision:", precision)
print("Recall   :", recall)
print("F1 Score :", f1)
print("FPR      :", fpr)
print("Confusion Matrix:", tn, fp, fn, tp)
print("Prediction distribution:", np.unique(y_pred, return_counts=True))

# ============================================================
# SAVE MINIMAL RESULTS (LOW DISK SAFE)
# ============================================================

results = {
    "dataset": "PrimeVul",
    "model": "CodeBERT",
    "epochs": 6,
    "accuracy": float(accuracy),
    "precision": float(precision),
    "recall": float(recall),
    "f1": float(f1),
    "fpr": float(fpr),
    "confusion_matrix": {
        "tn": int(tn),
        "fp": int(fp),
        "fn": int(fn),
        "tp": int(tp)
    },
    "timestamp": datetime.now().strftime("%Y-%m-%d %H:%M:%S")
}

out_path = "/kaggle/working/CodeBERT_PrimeVul_metrics.json"
with open(out_path, "w") as f:
    json.dump(results, f)

print("\nResults saved to:", out_path)


Device: cuda
GPU: Tesla T4

Files in dataset:
 - primevul_test_paired.jsonl
 - primevul_train_paired.jsonl
 - primevul_valid_paired.jsonl

Loading PrimeVul dataset...
Train: (7578, 2)
Val  : (960, 2)
Test : (870, 2)

Train label distribution:
 label
1    3789
0    3789
Name: count, dtype: int64


tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/150 [00:00<?, ?B/s]

...


Starting CodeBERT training on PrimeVul...


/usr/local/lib/python3.12/dist-packages/torch/nn/parallel/_functions.py:71: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(


Step,Training Loss
100,0.697700
200,0.696600
300,0.697500
400,0.696700
500,0.694900
600,0.694500
700,0.695600
800,0.694800
900,0.695200
1000,0.694800



Evaluating on test set...



===== FINAL CODEBERT PRIMEVUL RESULTS =====
Accuracy : 0.5080459770114942
Precision: 0.5660377358490566
Recall   : 0.06896551724137931
F1 Score : 0.12295081967213115
FPR      : 0.052873563218390804
Confusion Matrix: 412 23 405 30
Prediction distribution: (array([0, 1]), array([817,  53]))

Results saved to: /kaggle/working/CodeBERT_PrimeVul_metrics.json
